In [3]:
from wrfpp import *

In [ ]:
def __getitem__(self, species: list[str], total: bool = True, *args):
    """Convert a set of aerosols mass concentration variables into a single Dataset with a 'bins' dimension.

    Parameters
    ----------
    species : list[str]
        The list of species of interest as called in MOSAIC chemistry outputs.
        For example, for na_aXX and na_cwXX, the user should provide "na".

    total : bool, optional
        This option defines whether the total value (non-activated + activated) is given for each bin, by default True.
        If set to false, the activated and non-activated aerosols are considered as independent species.

    *args: slice
            Slice of interest in the WRF output.

    Returns
    -------
    xr.Dataset
        The dataset with a 'bins' dimension.
    """

    wrf = self._dataset.wrf
    ds_binned = xr.Dataset()

    # Get number of bins for given output
    nbins = wrf.aer_nbins.__getitem__(*args).values
    # Loop over the species
    for species_name in species:
        # Search for binned variables for var_name
        pattern_a = re.compile(species_name + "_a[0-9]+")
        pattern_cw = re.compile(species_name + "_cw[0-9]+")
        matching_variables_names_a = [
            var_name
            for var_name in wrf.variables
            if pattern_a.fullmatch(var_name)
        ]
        matching_variables_names_cw = [
            var_name
            for var_name in wrf.variables
            if pattern_cw.fullmatch(var_name)
        ]

        # Test result for a search
        if matching_variables_names_a:
            # Check units
            expected_units = "ug/kg-dryair"
            for var_name in matching_variables_names_a:
                wrf.check_units(var_name, expected_units)

            # Convert a0X output for this species to one array with new bin dimension
            array_a = wrf[matching_variables_names_a].to_array(dim="bin")
            array_a["bin"] = np.arange(1, nbins + 1)
        # Raise aer key output not found
        else:
            msg = var_name + " variable not found for non-activated aerosols"
            raise KeyError(msg)
        # Test result for cw search
        if matching_variables_names_cw:
            # Check units
            expected_units = "ug/kg-dryair"
            for var_name in matching_variables_names_cw:
                wrf.check_units(var_name, expected_units)

            # Convert cw0X output for this species to one array with new bin dimension
            array_cw = wrf[matching_variables_names_cw].to_array(dim="bin")
            array_cw["bin"] = np.arange(1, nbins + 1)
        # Raise aer key output not found
        else:
            msg = var_name + " variable not found for activated aerosols"
            raise KeyError(msg)

        # Add new variables to ds_binned according to total option
        if total:
            ds_binned[species_name] = array_a + array_cw

        else:
            ds_binned[species_name + "_a"] = array_a
            ds_binned[species_name + "_cw"] = array_cw

    # Add bin size coordinates for bin dimension
    bins_charac_array = wrf.aer_bins_charac.values
    ds_binned = ds_binned.assign_coords(
        {
            "Dlo": ("bin", bins_charac_array[:, 0]),
            "Dmid": ("bin", bins_charac_array[:, 1]),
            "Dhi": ("bin", bins_charac_array[:, 2]),
            "dD": (
                "bin",
                bins_charac_array[:, 2] - bins_charac_array[:, 0],
            ),
        }
    )

    return xr.Dataset(
        ds_binned,
        attrs=dict(
            name="Subset of WRF aerosols' mass concentrations distributed over size bins",
            units="ug/kg-dryair",
        ),
    )

In [13]:
ds = open_dataset("/home/cldmtb/PhD/code/WRF-infra/postprocess/wrfout.nc")

__getitem__(self=ds, species=["na"], total=False)

<xarray.Dataset> Size: 15MB
Dimensions:  (Time: 1, south_north: 99, west_east: 99, bin: 4, bottom_top: 49)
Coordinates:
    XTIME    (Time) datetime64[ns] 8B ...
    XLAT     (Time, south_north, west_east) float32 39kB ...
    XLONG    (Time, south_north, west_east) float32 39kB ...
  * bin      (bin) int64 32B 1 2 3 4
    Dlo      (bin) float32 16B 3.906e-08 1.562e-07 6.25e-07 2.5e-06
    Dmid     (bin) float32 16B 7.812e-08 3.125e-07 1.25e-06 5e-06
    Dhi      (bin) float32 16B 1.562e-07 6.25e-07 2.5e-06 1e-05
    dD       (bin) float32 16B 1.172e-07 4.687e-07 1.875e-06 7.5e-06
Dimensions without coordinates: Time, south_north, west_east, bottom_top
Data variables:
    na_a     (bin, Time, bottom_top, south_north, west_east) float32 8MB 0.02...
    na_cw    (bin, Time, bottom_top, south_north, west_east) float32 8MB 0.0 ...
Attributes:
    name:       Subset of binned aerosol mass concentrations
    long_name:  Subset of WRF aerosols' mass concentrations distributed over ...
    units:      ug/kg-dryair

In [ ]:
def __getitem__(self, species: list[str], total: bool = True, *args):
    """Convert a set of aerosols mass concentration variables into a single Dataset with a 'bins' dimension.

    Parameters
    ----------
    species : list[str]
        The list of species of interest as called in MOSAIC chemistry outputs.
        For example, for na_aXX and na_cwXX, the user should provide "na".

    total : bool, optional
        This option defines whether the total value (non-activated + activated) is given for each bin, by default True.
        If set to false, the activated and non-activated aerosols are considered as independent species.

    *args: slice
            Slice of interest in the WRF output.

    Returns
    -------
    xr.Dataset
        The dataset with a 'bins' dimension.
    """

    ds = self._dataset.wrf
    ds_binned = xr.Dataset()

    # Get number of bins for given output
    nbins = ds.wrf.aer_nbins.__getitem__(*args).values
    # Loop over the species
    for species_name in species:
        # Search for binned variables for var_name
        pattern_a = re.compile(species_name + "_a[0-9]+")
        pattern_cw = re.compile(species_name + "_cw[0-9]+")
        matching_variables_names_a = [
            var_name
            for var_name in ds.variables
            if pattern_a.fullmatch(var_name)
        ]
        matching_variables_names_cw = [
            var_name
            for var_name in ds.variables
            if pattern_cw.fullmatch(var_name)
        ]

        # Test result for a search
        if matching_variables_names_a:
            # Check units
            expected_units = "ug/kg-dryair"
            for var_name in matching_variables_names_a:
                ds.wrf.check_units(var_name, expected_units)

            # Convert a0X output for this species to one array with new bin dimension
            array_a = ds[matching_variables_names_a].to_array(dim="bin")
            array_a["bin"] = np.arange(1, nbins + 1)
        # Raise aer key output not found
        else:
            msg = var_name + " variable not found for non-activated aerosols"
            raise KeyError(msg)
        # Test result for cw search
        if matching_variables_names_cw:
            # Check units
            expected_units = "ug/kg-dryair"
            for var_name in matching_variables_names_cw:
                ds.wrf.check_units(var_name, expected_units)

            # Convert cw0X output for this species to one array with new bin dimension
            array_cw = ds[matching_variables_names_cw].to_array(dim="bin")
            array_cw["bin"] = np.arange(1, nbins + 1)
        # Raise aer key output not found
        else:
            msg = var_name + " variable not found for activated aerosols"
            raise KeyError(msg)

        # Add new variables to ds_binned according to total option
        if total:
            ds_binned[species_name] = array_a + array_cw

        else:
            ds_binned[species_name + "_a"] = array_a
            ds_binned[species_name + "_cw"] = array_cw

    # Add bin size coordinates for bin dimension
    bins_charac_array = ds.wrf.aer_bins_charac.values
    ds_binned = ds_binned.assign_coords(
        {
            "Dlo": ("bin", bins_charac_array[:, 0]),
            "Dmid": ("bin", bins_charac_array[:, 1]),
            "Dhi": ("bin", bins_charac_array[:, 2]),
            "dD": (
                "bin",
                bins_charac_array[:, 2] - bins_charac_array[:, 0],
            ),
        }
    )

    return xr.Dataset(
        ds_binned.__getitem__(*args),
        name="Subset of binned aerosol mass concentrations",
        attrs=dict(
            long_name="Subset of WRF aerosols' mass concentrations distributed over size bins",
            units="ug/kg-dryair",
        ),
    )

four size bins (0.039–0.156, 0.156–0.625, 0.625–2.500, and 2.5–10.0 µm dry diameters

dlo_1 = 3.90625e-8 # First bin lower diameter in meters (3.90625 nm)

dhi_n = 10.0e-6 # Last bin upper diameter in meters (10 µm)


In [5]:
import numpy as np


In [6]:
self = ds

nbins = _getnbins_(self)

In [ ]:
def __getitem__(self, species: list[str], total: bool = True, *args):
    """Convert a set of aerosols mass concentration variables into a single Dataset with a 'bins' dimension.

    Parameters
    ----------
    species : list[str]
        The list of species of interest as called in MOSAIC chemistry outputs.
        For example, for na_aXX and na_cwXX, the user should provide "na".

    total : bool, optional
        This option defines whether the total value (non-activated + activated) is given for each bin, by default True.
        If set to false, the activated and non-activated aerosols are considered as independent species.

    *args: slice
            Slice of interest in the WRF output.

    Returns
    -------
    xr.Dataset
        The dataset with a 'bins' dimension.
    """

    ds = self._dataset.wrf
    ds_binned = xr.Dataset()

    # Get number of bins for given output
    nbins = ds.wrf.aer_nbins.__getitem__(*args).values
    # Loop over the species
    for species_name in species:
        # Search for binned variables for var_name
        pattern_a = re.compile(species_name + "_a[0-9]+")
        pattern_cw = re.compile(species_name + "_cw[0-9]+")
        matching_variables_names_a = [
            var_name
            for var_name in ds.variables
            if pattern_a.fullmatch(var_name)
        ]
        matching_variables_names_cw = [
            var_name
            for var_name in ds.variables
            if pattern_cw.fullmatch(var_name)
        ]

        # Test result for a search
        if matching_variables_names_a:
            # Check units
            expected_units = "ug/kg-dryair"
            for var_name in matching_variables_names_a:
                ds.wrf.check_units(var_name, expected_units)

            # Convert a0X output for this species to one array with new bin dimension
            array_a = ds[matching_variables_names_a].to_array(dim="bin")
            array_a["bin"] = np.arange(1, nbins + 1)
        # Raise aer key output not found
        else:
            msg = var_name + " variable not found for non-activated aerosols"
            raise KeyError(msg)
        # Test result for cw search
        if matching_variables_names_cw:
            # Check units
            expected_units = "ug/kg-dryair"
            for var_name in matching_variables_names_cw:
                ds.wrf.check_units(var_name, expected_units)

            # Convert cw0X output for this species to one array with new bin dimension
            array_cw = ds[matching_variables_names_cw].to_array(dim="bin")
            array_cw["bin"] = np.arange(1, nbins + 1)
        # Raise aer key output not found
        else:
            msg = var_name + " variable not found for activated aerosols"
            raise KeyError(msg)

        # Add new variables to ds_binned according to total option
        if total:
            ds_binned[species_name] = array_a + array_cw

        else:
            ds_binned[species_name + "_a"] = array_a
            ds_binned[species_name + "_cw"] = array_cw

    # Add bin size coordinates for bin dimension
    bins_charac_array = ds.wrf.aer_bins_charac.values
    ds_binned = ds_binned.assign_coords(
        {
            "Dlo": ("bin", bins_charac_array[:, 0]),
            "Dmid": ("bin", bins_charac_array[:, 1]),
            "Dhi": ("bin", bins_charac_array[:, 2]),
            "dD": (
                "bin",
                bins_charac_array[:, 2] - bins_charac_array[:, 0],
            ),
        }
    )

    return xr.Dataset(
        ds_binned.__getitem__(*args),
        name="Subset of binned aerosol mass concentrations",
        attrs=dict(
            long_name="Subset of WRF aerosols' mass concentrations distributed over size bins",
            units="ug/kg-dryair",
        ),
    )

In [7]:
def _get_mosaic_size_bins_(self) -> list[list[float]]:
    """Return the bins' characteristics for the size distributions of the number concentration of aerosols.

    Return
    ------
    list[list[float]]

        The list of each bin's characteristics [lower, center, higher] for the size distribution of the number concentration of aerosols.

    """

    # Constants (from WRF-Chem logic)

    dlower_1 = 3.90625e-8  # First bin lower diameter in meters (0.0390625 nm)

    dhigher_n = 10.0e-6  # Last bin upper diameter in meters (10 µm)

    # Get number of bins for given output

    nbins = _getnbins_(self)

    # Calculate logarithmic spacing factor

    log_step = np.log(dhigher_n / dlower_1) / nbins

    # Initialize arrays

    dlower = np.zeros(shape=nbins, dtype=np.float32)

    dcenter = np.zeros(shape=nbins, dtype=np.float32)

    dhigher = np.zeros(shape=nbins, dtype=np.float32)

    # Set first lower diameter and last upper diameter

    dlower[0] = dlower_1

    dhigher[-1] = dhigher_n

    # Compute bin edges

    for n in range(1, nbins):
        dlower[n] = dlower[0] * np.exp(n * log_step)

        dhigher[n - 1] = dlower[n]

    # Compute bin centers

    for n in range(nbins):
        dcenter[n] = np.sqrt(dlower[n] * dhigher[n])

    # Convert from meters to microns (µm)

    dlower_um = dlower * 1e6

    dcenter_um = dcenter * 1e6

    dhigher_um = dhigher * 1e6

    # Define output edge list

    bins_charac = [
        [dlower_um_ii, dcenter_um_ii, dhigher_um_ii]
        for dlower_um_ii, dcenter_um_ii, dhigher_um_ii in zip(
            dlower_um, dcenter_um, dhigher_um
        )
    ]

    return bins_charac

In [8]:
_get_mosaic_size_bins_(self)

[[np.float32(0.0390625), np.float32(0.078125), np.float32(0.15625)],
 [np.float32(0.15625), np.float32(0.3125), np.float32(0.625)],
 [np.float32(0.625), np.float32(1.25), np.float32(2.5)],
 [np.float32(2.5), np.float32(5.0), np.float32(10.0)]]

In [9]:
wrf = self._dataset.wrf

list_of_variables = list(wrf.keys())


list_of_aer_variables = [
    var_name.split("_a")[0]
    for var_name in list_of_variables
    if "na_a" in var_name
]

In [10]:
wrf = ds.wrf

nbins = wrf.aer_nbins

str(nbins.values)

'4'

In [11]:
list_of_variables_filtered = [
    var_name for var_name in list_of_variables if "_a0" in var_name
]

In [12]:
list_of_variables_filtered

['so4_a01',
 'no3_a01',
 'cl_a01',
 'msa_a01',
 'nh4_a01',
 'na_a01',
 'oin_a01',
 'oc_a01',
 'bc_a01',
 'hysw_a01',
 'water_a01',
 'num_a01',
 'so4_a02',
 'no3_a02',
 'cl_a02',
 'msa_a02',
 'nh4_a02',
 'na_a02',
 'oin_a02',
 'oc_a02',
 'bc_a02',
 'hysw_a02',
 'water_a02',
 'num_a02',
 'so4_a03',
 'no3_a03',
 'cl_a03',
 'msa_a03',
 'nh4_a03',
 'na_a03',
 'oin_a03',
 'oc_a03',
 'bc_a03',
 'hysw_a03',
 'water_a03',
 'num_a03',
 'so4_a04',
 'no3_a04',
 'cl_a04',
 'msa_a04',
 'nh4_a04',
 'na_a04',
 'oin_a04',
 'oc_a04',
 'bc_a04',
 'hysw_a04',
 'water_a04',
 'num_a04']

In [13]:
species = ["na", "bc"]

aname = species + "_a"

species + "_cw"


TypeError: can only concatenate list (not "str") to list

In [14]:
import re

In [ ]:
bins_charac_array = ds.wrf.aer_bins_charac.values

bins_charac_array[:, 0]

array([3.90625e-08, 1.56250e-07, 6.25000e-07, 2.50000e-06], dtype=float32)

In [8]:
def convert_set_of_aer_variables_to_binned_dataset(
    self, species: list[str], total: bool = True, *args
):
    """Convert a set of aerosols mass concentration variables into a single Dataset with a 'bins' dimension.

    Parameters
    ----------
    species : list[str]
        The list of species of interest as called in MOSAIC chemistry outputs.
        For example, for na_aXX and na_cwXX, the user should provide "na".

    total : bool, optional
        This option defines whether the total value (non-activated + activated) is given for each bin, by default True.
        If set to false, the activated and non-activated aerosols are considered as independent species.

    *args: slice
            Slice of interest in the WRF output.

    Returns
    -------
    xr.Dataset
        The dataset with a 'bins' dimension.
    """

    ds = self._dataset
    ds_binned = xr.Dataset()

    # Get number of bins for given output
    nbins = ds.wrf.aer_nbins.__getitem__(*args).values
    # Loop over the species
    for species_name in species:
        # Search for binned variables for var_name
        pattern_a = re.compile(species_name + "_a[0-9]+")
        pattern_cw = re.compile(species_name + "_cw[0-9]+")
        matching_variables_names_a = [
            var_name
            for var_name in ds.variables
            if pattern_a.fullmatch(var_name)
        ]
        matching_variables_names_cw = [
            var_name
            for var_name in ds.variables
            if pattern_cw.fullmatch(var_name)
        ]

        # Test result for a search
        if matching_variables_names_a:
            # Check units
            expected_units = "ug/kg-dryair"
            for var_name in matching_variables_names_a:
                ds.wrf.check_units(var_name, expected_units)

            # Convert a0X output for this species to one array with new bin dimension
            array_a = ds[matching_variables_names_a].to_array(dim="bin")
            array_a["bin"] = np.arange(1, nbins + 1)
        # Raise aer key output not found
        else:
            msg = var_name + " variable not found for non-activated aerosols"
            raise KeyError(msg)
        # Test result for cw search
        if matching_variables_names_cw:
            # Check units
            expected_units = "ug/kg-dryair"
            for var_name in matching_variables_names_cw:
                ds.wrf.check_units(var_name, expected_units)

            # Convert cw0X output for this species to one array with new bin dimension
            array_cw = ds[matching_variables_names_cw].to_array(dim="bin")
            array_cw["bin"] = np.arange(1, nbins + 1)
        # Raise aer key output not found
        else:
            msg = var_name + " variable not found for activated aerosols"
            raise KeyError(msg)

        # Add new variables to ds_binned according to total option
        if total:
            ds_binned[species_name] = array_a + array_cw

        else:
            ds_binned[species_name + "_a"] = array_a
            ds_binned[species_name + "_cw"] = array_cw

    # Add bin size coordinates for bin dimension
    bins_charac_array = ds.wrf.aer_bins_charac.values
    ds_binned = ds_binned.assign_coords(
        {
            "Dlo": ("bin", bins_charac_array[:, 0]),
            "Dmid": ("bin", bins_charac_array[:, 1]),
            "Dhi": ("bin", bins_charac_array[:, 2]),
            "dD": (
                "bin",
                bins_charac_array[:, 2] - bins_charac_array[:, 0],
            ),
        }
    )

    return ds_binned

In [9]:
convert_set_of_aer_variables_to_binned_dataset(
    self=open_dataset("/home/cldmtb/PhD/code/WRF-infra/postprocess/wrfout.nc"),
    species=["na"],
)

<xarray.Dataset> Size: 8MB
Dimensions:  (Time: 1, south_north: 99, west_east: 99, bin: 4, bottom_top: 49)
Coordinates:
    XTIME    (Time) datetime64[ns] 8B 2020-04-01
    XLAT     (Time, south_north, west_east) float32 39kB 29.46 29.9 ... 35.98
    XLONG    (Time, south_north, west_east) float32 39kB -66.36 -65.77 ... 105.8
  * bin      (bin) int64 32B 1 2 3 4
    Dlo      (bin) float32 16B 3.906e-08 1.562e-07 6.25e-07 2.5e-06
    Dmid     (bin) float32 16B 7.812e-08 3.125e-07 1.25e-06 5e-06
    Dhi      (bin) float32 16B 1.562e-07 6.25e-07 2.5e-06 1e-05
    dD       (bin) float32 16B 1.172e-07 4.687e-07 1.875e-06 7.5e-06
Dimensions without coordinates: Time, south_north, west_east, bottom_top
Data variables:
    na       (bin, Time, bottom_top, south_north, west_east) float32 8MB 0.02...